<a href="https://colab.research.google.com/github/danny-lin714/AI-Security-Learning-Path/blob/main/Day02_Backprop_micrograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 2：Backpropagation：讓梯度自己往回傳

**AI Security 30 天入門計畫 · 第 1 週 · 2026-11-05**

昨天你寫出了 `Value` 類別，也手動算出每個節點的梯度。今天把「手動」換成「自動」：讓每個運算記住自己的局部導數，用 chain rule 把梯度從輸出一路傳回輸入，最後用這套 engine 訓練一個小 MLP。

| 項目 | 內容 |
|---|---|
| 讀／看 | Karpathy《Neural Networks: Zero to Hero》第 1 集（micrograd）後半 |
| 動手做 | 完成 micrograd 的自動 backward，用它訓練一個小 MLP |
| 產出 | 一段筆記：chain rule 怎麼讓梯度往回傳 |

### 今天的流程（約 2.5 小時）

| 時間 | 段落 |
|---|---|
| 60–75 分鐘 | 看影片後半：從「把 `_backward` 寫進每個運算」那段開始，看到最後的訓練迴圈 |
| 40–60 分鐘 | Part 1–3：完成 `Value` 的反向傳播，通過測試，畫出計算圖 |
| 30–40 分鐘 | Part 4–5：Neuron、MLP、訓練迴圈 |
| 15 分鐘 | Part 6：對輸入求梯度（第 2 週的預告） |
| 20 分鐘 | Part 7：寫筆記，commit 到 GitHub |

### 怎麼用這份 notebook

- 標 `TODO` 的地方由你填。寫完後執行下面的測試格，全部 ✅ 再往下走。
- 每個 TODO 後面有一格「參考解答」，程式碼預設隱藏。卡住時，勾選該格的 `USE_REFERENCE` 再執行，它會用參考版本覆蓋你的實作，讓後面的段落能繼續。沒勾選時執行它不會有任何效果，所以 Run all 也安全。雙擊那一格可以展開程式碼。
- 今天全部是純 Python，**不需要 GPU**，Colab 預設的 CPU runtime 就好。

影片：https://www.youtube.com/watch?v=VMj-3S1tku0
原始碼：https://github.com/karpathy/micrograd

In [ ]:
# @title 環境設定（直接執行）
import math
import random
import sys

import matplotlib.pyplot as plt
import torch

random.seed(42)            # 固定 random seed，讓結果可以重現
sys.setrecursionlimit(10_000)  # backward() 用遞迴做拓撲排序，計算圖大一點時需要更深的遞迴

print("PyTorch", torch.__version__, "（今天只拿來對答案）")

## Part 1：讓每個運算知道怎麼往回傳

### 核心想法

假設 `out = f(a, b)`，最終的 loss 是 $L$。反向傳播走到 `out` 時，我們已經知道 $\frac{\partial L}{\partial \text{out}}$，也就是 `out.grad`。根據 chain rule：

$$
\frac{\partial L}{\partial a} = \frac{\partial L}{\partial \text{out}} \cdot \frac{\partial \text{out}}{\partial a}
$$

所以每個運算只需要知道自己的**局部導數** $\frac{\partial \text{out}}{\partial a}$。它不需要知道整張計算圖長什麼樣，也不需要知道 loss 是什麼。寫成程式就是：

```python
self.grad += 局部導數 * out.grad
```

### 局部導數表

| 運算 | forward | 對 `self` 的局部導數 | 對 `other` 的局部導數 |
|---|---|---|---|
| 加法 | `a + b` | $1$ | $1$ |
| 乘法 | `a * b` | $b$ | $a$ |
| 次方（k 是常數） | `a ** k` | $k \cdot a^{k-1}$ | — |
| exp | `a.exp()` | $e^a$（就是 `out.data`） | — |
| tanh | `a.tanh()` | $1 - \tanh^2(a)$ | — |
| ReLU | `a.relu()` | $a > 0$ 時為 1，否則為 0 | — |

減法、負號和除法不用另外寫 backward，因為它們都能用上面的運算組出來，例如 `a / b = a * b**-1`。

### 為什麼是 `+=` 而不是 `=`？

同一個節點如果被用了兩次（例如 `b = a + a`），它會從兩條路收到梯度，總梯度是兩條路的和（multivariate chain rule）。用 `=` 的話，第二條路會把第一條路的梯度蓋掉。這就是影片裡 Karpathy 遇到的 bug，Part 2 有測試專門抓它。

### `backward()` 要做的三件事

1. **拓撲排序**：把計算圖排成一列，保證每個節點都排在「用到它的節點」之前。
2. **設定起點**：$\frac{\partial L}{\partial L} = 1$，所以 `self.grad = 1.0`。
3. **反向走一遍**：照拓撲排序的反向順序呼叫每個節點的 `_backward()`。輪到某個節點時，所有用到它的節點都已經把梯度傳給它了，所以它的 `grad` 已經完整。

In [ ]:
class Value:
    """一個純量節點：存數值 data、梯度 grad，並記得自己是由哪些節點、哪個運算算出來的。"""

    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0                  # ∂L/∂(這個節點)，一開始是 0
        self._backward = lambda: None    # 葉節點（輸入、權重）沒有東西要往回傳
        self._prev = set(_children)      # 算出這個節點的輸入節點
        self._op = _op                   # 產生這個節點的運算，畫圖用
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    # ========== 有自己 backward 的基本運算 ==========

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
            pass

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
            pass

        out._backward = _backward
        return out

    def __pow__(self, k):
        assert isinstance(k, (int, float)), "只支援常數次方"
        out = Value(self.data ** k, (self,), f'**{k}')

        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad
            # TODO: 局部導數是 k * self.data ** (k - 1)
            pass

        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), (self,), 'exp')

        def _backward():
            self.grad = math.exp(self.data) * out.grad
            pass

        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')

        def _backward():
            # TODO: 局部導數是 1 - t**2
            pass

        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), 'ReLU')

        def _backward():
            # TODO: 輸出大於 0 時局部導數是 1，否則是 0
            pass

        out._backward = _backward
        return out

    # ========== 反向傳播 ==========

    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
          if v not in visited:
            visited.add(v)
            for child in v._prev:
              build_topo(child)
            topo.append(v)
            # TODO: 如果 v 還沒拜訪過：標記為已拜訪 → 先對 v._prev 裡每個節點遞迴 → 最後把 v 加進 topo
            pass

        build_topo(self)
        # TODO: 設定起點的梯度
        # TODO: 依 topo 的「反向」順序，呼叫每個節點的 _backward()
        raise NotImplementedError("完成 backward() 的 TODO 後，把這行刪掉")

    # ========== 用上面的運算組出來的（不用寫 backward） ==========

    def __neg__(self):              # -self
        return self * -1

    def __sub__(self, other):       # self - other
        return self + (-other)

    def __rsub__(self, other):      # other - self，other 是一般數字時
        return other + (-self)

    def __radd__(self, other):      # other + self
        return self + other

    def __rmul__(self, other):      # other * self
        return self * other

    def __truediv__(self, other):   # self / other
        return self * other ** -1

    def __rtruediv__(self, other):  # other / self
        return other * self ** -1

In [ ]:
# @title 參考解答：Value（卡住時勾選 USE_REFERENCE 再執行） { display-mode: "form" }
USE_REFERENCE = True  # @param {type:"boolean"}

if USE_REFERENCE:
    class Value:
        def __init__(self, data, _children=(), _op='', label=''):
            self.data = data
            self.grad = 0.0
            self._backward = lambda: None
            self._prev = set(_children)
            self._op = _op
            self.label = label

        def __repr__(self):
            return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

        def __add__(self, other):
            other = other if isinstance(other, Value) else Value(other)
            out = Value(self.data + other.data, (self, other), '+')
            def _backward():
                self.grad += 1.0 * out.grad
                other.grad += 1.0 * out.grad
            out._backward = _backward
            return out

        def __mul__(self, other):
            other = other if isinstance(other, Value) else Value(other)
            out = Value(self.data * other.data, (self, other), '*')
            def _backward():
                self.grad += other.data * out.grad
                other.grad += self.data * out.grad
            out._backward = _backward
            return out

        def __pow__(self, k):
            assert isinstance(k, (int, float)), "只支援常數次方"
            out = Value(self.data ** k, (self,), f'**{k}')
            def _backward():
                self.grad += k * self.data ** (k - 1) * out.grad
            out._backward = _backward
            return out

        def exp(self):
            out = Value(math.exp(self.data), (self,), 'exp')
            def _backward():
                self.grad += out.data * out.grad
            out._backward = _backward
            return out

        def tanh(self):
            t = math.tanh(self.data)
            out = Value(t, (self,), 'tanh')
            def _backward():
                self.grad += (1 - t ** 2) * out.grad
            out._backward = _backward
            return out

        def relu(self):
            out = Value(max(0.0, self.data), (self,), 'ReLU')
            def _backward():
                self.grad += (1.0 if out.data > 0 else 0.0) * out.grad
            out._backward = _backward
            return out

        def backward(self):
            topo, visited = [], set()
            def build_topo(v):
                if v not in visited:
                    visited.add(v)
                    for child in v._prev:
                        build_topo(child)
                    topo.append(v)
            build_topo(self)
            self.grad = 1.0
            for v in reversed(topo):
                v._backward()

        def __neg__(self): return self * -1
        def __sub__(self, other): return self + (-other)
        def __rsub__(self, other): return other + (-self)
        def __radd__(self, other): return self + other
        def __rmul__(self, other): return self * other
        def __truediv__(self, other): return self * other ** -1
        def __rtruediv__(self, other): return other * self ** -1

    print("已改用參考版本的 Value")

## Part 2：檢查你的梯度對不對

用三方對答案：

1. **你的 micrograd**
2. **PyTorch autograd**：同一個式子交給 PyTorch 算，兩邊的值和梯度應該一樣。
3. **數值微分**：Day 1 的導數定義 $\frac{f(x+h) - f(x-h)}{2h}$，只在出錯時顯示，幫你判斷是 forward 還是 backward 寫錯。

小技巧：`torch.Tensor` 也有 `+`、`*`、`**`、`.tanh()`、`.exp()`、`.relu()`，所以同一個 Python 函式可以同時餵 `Value` 和 tensor。

In [ ]:
# @title 測試工具（直接執行）
def _close(a, b, tol=1e-6):
    return math.isclose(a, b, rel_tol=tol, abs_tol=tol)

def check_grads(name, f, inputs):
    """比較 f 在 inputs 上的值與梯度：你的 micrograd vs PyTorch，出錯時附上數值微分。"""
    vs = [Value(x) for x in inputs]
    try:
        out = f(*vs)
        out.backward()
    except NotImplementedError as e:
        print(f"⏳ {name}：{e}")
        return False
    mine = [v.grad for v in vs]

    ts = [torch.tensor(x, dtype=torch.float64, requires_grad=True) for x in inputs]
    t_out = f(*ts)
    t_out.backward()
    ref = [t.grad.item() for t in ts]

    ok = _close(out.data, t_out.item()) and all(_close(m, r) for m, r in zip(mine, ref))
    print(f"{'✅' if ok else '❌'} {name}")
    if not ok:
        h = 1e-6
        print(f"     forward    你的 {out.data:+.6f} | PyTorch {t_out.item():+.6f}")
        for i in range(len(inputs)):
            plus, minus = list(inputs), list(inputs)
            plus[i] += h
            minus[i] -= h
            num = (f(*[Value(x) for x in plus]).data - f(*[Value(x) for x in minus]).data) / (2 * h)
            print(f"     ∂out/∂x{i}  你的 {mine[i]:+.6f} | PyTorch {ref[i]:+.6f} | 數值微分 {num:+.6f}")
    return ok

In [ ]:
print("— 單一運算 —")
check_grads("加法  a + b", lambda a, b: a + b, [2.0, -3.0])
check_grads("乘法  a * b", lambda a, b: a * b, [2.0, -3.0])
check_grads("次方  a ** 3", lambda a: a ** 3, [1.5])
check_grads("exp   a.exp()", lambda a: a.exp(), [0.7])
check_grads("tanh  a.tanh()", lambda a: a.tanh(), [0.8])
check_grads("ReLU  輸入為正", lambda a: a.relu(), [2.0])
check_grads("ReLU  輸入為負", lambda a: a.relu(), [-2.0])

print("\n— 組合運算 —")
check_grads("影片裡的神經元 tanh(x1·w1 + x2·w2 + b)",
            lambda x1, x2, w1, w2, b: (x1 * w1 + x2 * w2 + b).tanh(),
            [2.0, 0.0, -3.0, 1.0, 6.8813735870195432])
check_grads("減法與除法 (a - b) / (a + b)", lambda a, b: (a - b) / (a + b), [3.0, 1.0])

print("\n— 同一個節點被用兩次（用 = 而不是 += 的話，這裡會失敗）—")
check_grads("a + a", lambda a: a + a, [3.0])
check_grads("(a·b)·(a + b)", lambda a, b: (a * b) * (a + b), [-2.0, 3.0])

print("\n— 大魔王：micrograd README 的範例 —")
def readme_expr(a, b):
    c = a + b
    d = a * b + b ** 3
    c = c + c + 1
    c = c + 1 + c + (-a)
    d = d + d * 2 + (b + a).relu()
    d = d + 3 * d + (b - a).relu()
    e = c - d
    f = e ** 2
    g = f / 2.0
    g = g + 10.0 / f
    return g
check_grads("README 範例（g ≈ 24.7041）", readme_expr, [-4.0, 2.0]);

— 單一運算 —


NameError: name 'check_grads' is not defined

### 小實驗：同一個函式，拆得多細都可以

影片裡 Karpathy 把 `tanh` 拆成 `exp`、加減和除法重新組一次，梯度完全一樣。這說明 backward 只要求「每個運算知道自己的局部導數」，運算要切多細由你決定。

注意下面的 `e` 被用了兩次，所以這個實驗也順便測了 `+=`。

In [ ]:
x = Value(0.5)
y = x.tanh()
y.backward()
print(f"直接用 tanh     ：dy/dx = {x.grad:.6f}")

x = Value(0.5)
e = (2 * x).exp()
y = (e - 1) / (e + 1)      # tanh(x) = (e^{2x} - 1) / (e^{2x} + 1)
y.backward()
print(f"拆成 exp 再組合 ：dy/dx = {x.grad:.6f}")

print(f"理論值 1 - tanh²：      {1 - math.tanh(0.5) ** 2:.6f}")

## Part 3：把計算圖畫出來

每個方框顯示節點的 `data` 和 `grad`。畫出來之後，挑一個乘法節點手算驗證一次：

> 輸入的 grad = 另一個輸入的 data × 輸出的 grad

這就是你在 `__mul__` 裡寫的那一行。

In [ ]:
# @title 畫圖工具（直接執行）
from graphviz import Digraph   # Colab 已內建；若出現 ImportError，先執行 !pip install graphviz

def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'})
    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        dot.node(uid, label=f"{{ {n.label} | data {n.data:.4f} | grad {n.grad:.4f} }}", shape='record')
        if n._op:
            dot.node(uid + n._op, label=n._op)
            dot.edge(uid + n._op, uid)
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot

In [ ]:
# 影片裡的神經元：o = tanh(x1·w1 + x2·w2 + b)
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
b = Value(6.8813735870195432, label='b')

x1w1 = x1 * w1; x1w1.label = 'x1*w1'
x2w2 = x2 * w2; x2w2.label = 'x2*w2'
s = x1w1 + x2w2; s.label = 'x1w1+x2w2'
n = s + b; n.label = 'n'
o = n.tanh(); o.label = 'o'

o.backward()
draw_dot(o)

**看圖回答**（寫在 Part 7 的筆記裡）：

1. 為什麼 `w2.grad` 是 0？從 chain rule 的角度說明。
2. `x2.data` 是 0，但 `x2.grad` 不是 0。為什麼？

## Part 4：用 `Value` 組出神經網路

- **Neuron**：$\tanh(w \cdot x + b)$。所有 $w$ 和 $b$ 都是 `Value`，所以算完 loss 後呼叫 `backward()`，它們的 `grad` 就是 $\frac{\partial L}{\partial w}$。
- **Layer**：好幾個 neuron 吃同一組輸入。
- **MLP**：好幾層串起來，前一層的輸出是下一層的輸入。

`parameters()` 會收集所有 $w$ 和 $b$，它們就是訓練時要更新的對象。你只需要寫 `Neuron`，`Layer` 和 `MLP` 已經寫好。

In [ ]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        out = sum (x*w for x, w in zip(x,self.w))+ b
        return out.tanh()

    def parameters(self):
      return self.w +[b]

In [ ]:
# @title 參考解答：Neuron（卡住時勾選 USE_REFERENCE 再執行） { display-mode: "form" }
USE_REFERENCE = True  # @param {type:"boolean"}

if USE_REFERENCE:
    class Neuron:
        def __init__(self, nin):
            self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
            self.b = Value(random.uniform(-1, 1))

        def __call__(self, x):
            act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
            return act.tanh()

        def parameters(self):
            return self.w + [self.b]

    print("已改用參考版本的 Neuron")

In [ ]:
class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs   # 只有一個輸出時直接回傳 Value，比較好用

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP:
    def __init__(self, nin, nouts):
        sizes = [nin] + nouts
        self.layers = [Layer(sizes[i], sizes[i + 1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0

In [ ]:
try:
    random.seed(0)
    neuron = Neuron(3)
    x = [1.0, -2.0, 0.5]
    out = neuron(x)
    expected = math.tanh(sum(w.data * xi for w, xi in zip(neuron.w, x)) + neuron.b.data)
    print("✅" if isinstance(out, Value) and _close(out.data, expected) else "❌", "Neuron 的 forward")
    print("✅" if len(neuron.parameters()) == 4 else "❌", "Neuron(3) 有 4 個參數（3 個 w + 1 個 b）")

    mlp = MLP(3, [4, 4, 1])
    n_params = len(mlp.parameters())
    print("✅" if n_params == 41 else "❌", f"MLP(3, [4, 4, 1]) 有 {n_params} 個參數（應為 4·4 + 4·5 + 1·5 = 41）")
except NotImplementedError as e:
    print("⏳", e)

## Part 5：訓練迴圈

資料集是影片裡的 4 筆資料，每筆 3 個特徵，目標是 $+1$ 或 $-1$。loss 用 mean squared error：

$$
L = \frac{1}{N} \sum_{i=1}^{N} \big(\hat{y}_i - y_i\big)^2
$$

每一步做五件事，順序很重要：

| 步驟 | micrograd | 明天的 PyTorch |
|---|---|---|
| 1. forward | `ypred = [model(x) for x in xs]` | `ypred = model(X)` |
| 2. loss | 自己寫 MSE | `loss = loss_fn(ypred, y)` |
| 3. 梯度歸零 | `model.zero_grad()` | `optimizer.zero_grad()` |
| 4. backward | `loss.backward()` | `loss.backward()` |
| 5. 更新參數 | `p.data -= lr * p.grad` | `optimizer.step()` |

這五步就是第 1 週檢查點要求你「不看筆記能寫出來」的訓練迴圈。今天先在 micrograd 上寫一次。

In [ ]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]

In [ ]:
def train(model, xs, ys, steps=100, lr=0.1, zero_grad=True):
    """用 gradient descent 訓練 model，回傳每一步的 loss（float 的 list）。"""
    losses = []
    for step in range(steps):
        # TODO 1｜forward：算出每筆資料的預測 ypred
        # TODO 2｜loss：MSE，也就是 (預測 − 目標)² 的平均
        # TODO 3｜zero_grad 為 True 時，把所有參數的梯度歸零
        # TODO 4｜backward
        # TODO 5｜每個參數往負梯度方向走一步：p.data -= lr * p.grad
        raise NotImplementedError("請完成 train()")
        losses.append(loss.data)
    return losses

In [ ]:
# @title 參考解答：train（卡住時勾選 USE_REFERENCE 再執行） { display-mode: "form" }
USE_REFERENCE = False  # @param {type:"boolean"}

if USE_REFERENCE:
    def train(model, xs, ys, steps=100, lr=0.1, zero_grad=True):
        losses = []
        for step in range(steps):
            ypred = [model(x) for x in xs]
            loss = sum((yp - yt) ** 2 for yp, yt in zip(ypred, ys)) / len(xs)
            if zero_grad:
                model.zero_grad()
            loss.backward()
            for p in model.parameters():
                p.data -= lr * p.grad
            losses.append(loss.data)
        return losses

    print("已改用參考版本的 train")

In [ ]:
random.seed(42)
model = MLP(3, [4, 4, 1])
print("參數數量：", len(model.parameters()))

losses = train(model, xs, ys, steps=100, lr=0.1)
print(f"loss：第 1 步 {losses[0]:.4f} → 第 {len(losses)} 步 {losses[-1]:.6f}")

plt.figure(figsize=(6, 3.5))
plt.plot(losses)
plt.yscale("log")
plt.xlabel("step")
plt.ylabel("MSE loss (log scale)")
plt.title("micrograd MLP(3, [4, 4, 1]) training")
plt.grid(alpha=0.3)
plt.show()

print("\n訓練後的預測：")
for x, yt in zip(xs, ys):
    print(f"  目標 {yt:+.0f}   預測 {model(x).data:+.4f}")

### 實驗：忘記 `zero_grad` 會怎樣？

影片最後 Karpathy 發現自己的訓練迴圈少了梯度歸零，訓練卻「看起來還行」。下面用同一組初始權重訓練兩次，一次有歸零、一次沒有，比較 loss 曲線和最後的權重大小。

In [ ]:
STEPS, LR = 100, 0.1

random.seed(42)
model_ok = MLP(3, [4, 4, 1])
random.seed(42)
model_bug = MLP(3, [4, 4, 1])   # 同一個 seed → 同一組初始權重

losses_ok = train(model_ok, xs, ys, steps=STEPS, lr=LR, zero_grad=True)
losses_bug = train(model_bug, xs, ys, steps=STEPS, lr=LR, zero_grad=False)

plt.figure(figsize=(6, 3.5))
plt.plot(losses_ok, label="with zero_grad")
plt.plot(losses_bug, label="without zero_grad (bug)")
plt.yscale("log")
plt.xlabel("step")
plt.ylabel("MSE loss (log scale)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

for name, m, ls in [("有歸零", model_ok, losses_ok), ("沒歸零", model_bug, losses_bug)]:
    w_max = max(abs(p.data) for p in m.parameters())
    g_max = max(abs(p.grad) for p in m.parameters())
    preds = ", ".join(f"{m(x).data:+.4f}" for x in xs)
    print(f"{name}：最後 loss {ls[-1]:.6f}｜最大 |w| {w_max:6.2f}｜最大 |grad| {g_max:.4f}｜預測 [{preds}]")

**想一想**：你大概會看到 bug 版本的 loss 反而降得更快、更低。這代表忘記歸零比較好嗎？沒歸零時，第 $t$ 步的 `p.grad` 其實是什麼？它和 Part 1 的 `+=` 有什麼關係？

<details>
<summary>提示（想過再點開）</summary>

`+=` 讓梯度一直累加，所以沒歸零時，第 $t$ 步的 `p.grad` 是前 $t$ 步所有梯度的總和。每一步走的距離是 `lr ×` 這個總和，只要方向大致一致，步伐就越來越大，像一顆沒有摩擦力、一路加速的球。

看印出來的數字：bug 版本的權重比正常版大很多，tanh 全部被推到飽和，輸出幾乎剛好是 ±1，所以 loss 接近 0。這是運氣：這個問題只有 4 筆資料，梯度方向一直沒變。飽和的 tanh 代表模型極度自信、局部導數接近 0，之後很難再修正；而累積的 `grad` 還留著一個很大的過時數字，代表參數仍在被往前推。資料一多、不同樣本的梯度方向互相衝突時，這些過時的梯度會把參數推往錯誤的方向。

教訓：loss 曲線好看不代表程式正確。第 2 週 Day 13 講 robustness 評估時，你會再遇到「數字很漂亮，但其實是評估寫錯」的情況。

</details>

## Part 6：對輸入求梯度（第 2 週預告）

同一次 `backward()`，計算圖裡**每一個** `Value` 都會拿到梯度，不只是權重。訓練時輸入 `x` 是一般的 float，沒有人保留它的梯度；把它包成 `Value`，就能拿到 $\frac{\partial L}{\partial x}$。

同一個梯度，兩種用途：

| | 改什麼 | 往哪個方向 | 目的 |
|---|---|---|---|
| 訓練 | 權重 $w$ | $w \leftarrow w - \text{lr} \cdot \frac{\partial L}{\partial w}$ | 讓 loss **下降** |
| 攻擊 | 輸入 $x$ | $x \leftarrow x + \varepsilon \cdot \operatorname{sign}\big(\frac{\partial L}{\partial x}\big)$ | 讓 loss **上升** |

下面這個就是 Day 8 會正式推導的 FGSM，只是目標換成今天訓練好的小 MLP。

In [1]:
def predict_and_loss(model, x_values, y_true):
    pred = model(x_values)
    return pred, (pred - y_true) ** 2

i = 0   # 拿第 0 筆資料來攻擊
x = [Value(v, label=f"x{j}") for j, v in enumerate(xs[i])]

model.zero_grad()
pred, loss = predict_and_loss(model, x, ys[i])
loss.backward()                 # 權重和輸入同時拿到梯度；這裡我們不更新權重
grad_x = [xj.grad for xj in x]

print(f"原始輸入 {xs[i]}，目標 {ys[i]:+.0f}，預測 {pred.data:+.4f}，loss {loss.data:.6f}")
print("∂L/∂x =", [round(g, 6) for g in grad_x])

sign = lambda g: (g > 0) - (g < 0)
print("\n  ε     預測      loss")
for eps in [0.0, 0.1, 0.25, 0.5, 1.0]:
    x_adv = [xj.data + eps * sign(g) for xj, g in zip(x, grad_x)]
    p_adv, l_adv = predict_and_loss(model, x_adv, ys[i])
    print(f"  {eps:<4}  {p_adv.data:+.4f}   {l_adv.data:.6f}")

NameError: name 'xs' is not defined

ε 越大，loss 越高，預測也越偏離目標。第 2 週你會把這件事搬到 CIFAR-10 的 CNN 上，並用 L∞ 範數限制「每個像素最多改多少」。

## Part 7：今天的產出

### 筆記：chain rule 怎麼讓梯度往回傳

用自己的話回答下面的問題，可以搭配 Part 3 的計算圖說明。寫完之後，這段筆記就是今天的產出。

1. 一個節點的 `_backward()` 只用到哪兩個數字？為什麼這樣就夠了？
2. 為什麼 `backward()` 一定要照拓撲排序的反向順序？順序錯了會發生什麼？
3. 為什麼是 `+=` 而不是 `=`？用 `a + a` 當例子。
4. 為什麼訓練時每一步都要 `zero_grad()`？它和第 3 題有什麼關係？
5. Part 3 的兩個看圖問題：`w2.grad` 為什麼是 0？`x2.grad` 為什麼不是 0？
6. 對權重求梯度和對輸入求梯度，用的是同一次 `backward()`，各自拿來做什麼？

（在這裡寫你的筆記）

### 存到 GitHub

1. Colab 選單 **檔案 → 在 GitHub 中儲存副本**（File → Save a copy in GitHub）。
2. 選你的作品集 repo，路徑建議 `week1/day02_backprop_micrograd.ipynb`。
3. Commit message 範例：`day02: autograd backward + train MLP with micrograd`
4. 勾選「Include a link to Colab」，之後從 GitHub 一鍵開回 Colab。

### 今日檢查清單

- [ ] Part 2 的測試全部 ✅（沒有用參考解答）
- [ ] Part 3 畫出計算圖，手算驗證一個乘法節點
- [ ] Part 5 的 loss 曲線降到 0.01 以下
- [ ] 完成 zero_grad 實驗，能解釋兩條曲線的差異
- [ ] Part 6 看到 ε 越大 loss 越高
- [ ] 筆記寫完，notebook commit 到 GitHub

## 加碼挑戰（選做）：二維分類與決策邊界

時間還夠的話，用同一套 micrograd 訓練一個真的二維分類器：sklearn 的 `make_moons` 資料集，100 個點、兩個彎月形的類別。這個問題不是直線分得開的，需要隱藏層。

- 訓練大約 1–3 分鐘（純 Python 很慢，這正是明天要學 PyTorch 的原因）。
- 試著改隱藏層大小、學習率、步數，看決策邊界怎麼變。
- 決策邊界圖可以放進你的作品集 README。

In [ ]:
from sklearn.datasets import make_moons
import numpy as np
import time

X, y = make_moons(n_samples=100, noise=0.1, random_state=1)
y = y * 2 - 1                            # 標籤從 0/1 改成 -1/+1，配合 tanh 的輸出範圍
moon_xs = [list(map(float, p)) for p in X]
moon_ys = [float(t) for t in y]

random.seed(1)
moon_model = MLP(2, [16, 16, 1])
print("參數數量：", len(moon_model.parameters()))

t0 = time.time()
moon_losses = train(moon_model, moon_xs, moon_ys, steps=100, lr=0.2)
acc = sum((moon_model(p).data > 0) == (t > 0) for p, t in zip(moon_xs, moon_ys)) / len(moon_ys)
print(f"訓練時間 {time.time() - t0:.0f} 秒，最後 loss {moon_losses[-1]:.4f}，訓練準確率 {acc:.0%}")

In [ ]:
# 決策邊界：在格點上逐點預測（約 5–20 秒）
h = 0.1
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
gx, gy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = np.array([moon_model([float(a), float(b)]).data for a, b in zip(gx.ravel(), gy.ravel())])
Z = Z.reshape(gx.shape)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(moon_losses)
axes[0].set_xlabel("step")
axes[0].set_ylabel("MSE loss")
axes[0].set_title("training loss")
axes[0].grid(alpha=0.3)

axes[1].contourf(gx, gy, Z > 0, cmap=plt.cm.Spectral, alpha=0.6)
axes[1].scatter(X[:, 0], X[:, 1], c=y, s=30, cmap=plt.cm.Spectral, edgecolors="k")
axes[1].set_title("decision boundary: micrograd MLP(2, [16, 16, 1])")
plt.tight_layout()
plt.show()